# Part 2 Control — Input Reconstruction vs Sufficient-State Encoding

Purpose: test whether the strong state-recovery result from Part 2 reflects genuine access to the Markov sufficient state, rather than generic reconstruction of visible input tokens from the last hidden state.

This control reuses the same pretrained Qwen2.5 models and same `Markov_Context_Paper/Part2/` project root.

## Control logic

For each \(k\in\{2,3,4\}\), compare:
- **visible-token reconstruction**: recover only the tokens actually present in the observed context;
- **full-state recovery**: recover the full \(k\)-token Markov state.

At \(L=k-1\), the oldest state token is hidden from the model. If full-state recovery is low while visible-token reconstruction is near-perfect, that supports the interpretation that the probe is decoding information actually present in the representation rather than hallucinating missing state.

We also add a **position-shuffled label control**: train the same probe on randomly permuted state labels. Performance should collapse to chance.

The published shuffled-label results are from the dedicated corrected ten-seed notebook/archive. Both training and validation labels are independently permuted; test labels remain true. Historical combined-control summaries are superseded for shuffled metrics.


## Execution contract
Start the kernel working directory at the repository root. Install `requirements-experiments.txt` and read `reproducibility/EXPERIMENTS.md` before Run All.
Use a separate working directory for reduced diagnostic runs. Full defaults are retained here; a reduced run does not reproduce publication statistics.
Record the environment and checkpoint revision for any new run with `python scripts/record_environment.py --output run-environment.json`. Historical checkpoint revisions were not recorded in the supplied archives.


In [ ]:

import os, sys, json, random, hashlib, gc, zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoModelForCausalLM

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

PROJECT_ROOT = Path.cwd() / "Markov_Context_Paper" / "Part2"
CONTROL_DIR = PROJECT_ROOT / "controls_input_vs_state"
EXPORTS_DIR = PROJECT_ROOT / "exports"
CSV_DIR = EXPORTS_DIR / "csv"
FIG_DIR = EXPORTS_DIR / "figures"
CACHE_DIR = PROJECT_ROOT / "hf_cache"

for p in [CONTROL_DIR, CSV_DIR, FIG_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MODELS = {
    "qwen05": "Qwen/Qwen2.5-0.5B",
    "qwen15": "Qwen/Qwen2.5-1.5B",
}
V=8
KS=[2,3,4]
SEEDS=list(range(10))
N_TRAIN=8000
N_VAL=2000
N_TEST=4000
BATCH_EXTRACT={"qwen05":256,"qwen15":128}
PROBE_BATCH=512
EPOCHS=20
LR=1e-3
PATIENCE=4

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def context_to_index(context,V):
    idx=torch.zeros(context.shape[:-1],dtype=torch.long,device=context.device)
    for j in range(context.shape[-1]): idx=idx*V+context[...,j]
    return idx

def generate_kernel(V,k,seed):
    g=torch.Generator(device=DEVICE); g.manual_seed(seed+10_000_019)
    logits=torch.randn((V**k,V),generator=g,device=DEVICE)
    return torch.softmax(logits,dim=-1)

@torch.no_grad()
def generate_dataset(P,V,k,L,n,seed,batch_size=1024):
    g=torch.Generator(device=DEVICE); g.manual_seed(seed+7_000_009)
    B=min(batch_size,n); hist_len=max(k,L)
    hist=torch.randint(0,V,(B,hist_len),generator=g,device=DEVICE)
    for _ in range(256):
        probs=P[context_to_index(hist[:,-k:],V)]
        y=torch.multinomial(probs,1,generator=g).squeeze(1)
        hist=torch.cat([hist[:,1:],y[:,None]],1)
    states=[]; obs=[]; ys=[]; remaining=n
    while remaining>0:
        take=min(B,remaining)
        h=hist[:take]
        states.append(h[:,-k:].cpu())
        obs.append(h[:,-L:].cpu())
        probs=P[context_to_index(h[:,-k:],V)]
        ys.append(torch.multinomial(probs,1,generator=g).squeeze(1).cpu())
        probs_all=P[context_to_index(hist[:,-k:],V)]
        y_all=torch.multinomial(probs_all,1,generator=g).squeeze(1)
        hist=torch.cat([hist[:,1:],y_all[:,None]],1)
        remaining-=take
    return torch.cat(states).long(), torch.cat(obs).long(), torch.cat(ys).long()

def choose_token_ids(model,V,seed):
    vocab=model.config.vocab_size
    rng=np.random.default_rng(seed+91337)
    candidates=np.arange(1000,max(1001,vocab-1000))
    chosen=rng.choice(candidates,size=V,replace=False)
    return torch.tensor(chosen,dtype=torch.long)

def map_symbols(x,mapping):
    return mapping[x]

def selected_final_layer(model):
    return int(model.config.num_hidden_layers)

@torch.no_grad()
def extract_final(model,input_ids,batch_size):
    reps=[]
    for i in range(0,len(input_ids),batch_size):
        batch=input_ids[i:i+batch_size].to(DEVICE)
        out=model(input_ids=batch,attention_mask=torch.ones_like(batch),
                  output_hidden_states=True,use_cache=False,return_dict=True)
        reps.append(out.hidden_states[-1][:,-1,:].detach().cpu().to(torch.float16))
        del out,batch
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return torch.cat(reps)


In [ ]:

class MultiTokenProbe(nn.Module):
    def __init__(self,d,npos,V):
        super().__init__()
        self.npos=npos; self.V=V
        self.head=nn.Linear(d,npos*V)
    def forward(self,x):
        return self.head(x).view(-1,self.npos,self.V)

def loss_multi(logits,y):
    return F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.reshape(-1))

def train_probe(Xtr,ytr,Xva,yva,V):
    model=MultiTokenProbe(Xtr.shape[1],ytr.shape[1],V).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    tr=DataLoader(TensorDataset(Xtr.float(),ytr),batch_size=PROBE_BATCH,shuffle=True)
    va=DataLoader(TensorDataset(Xva.float(),yva),batch_size=PROBE_BATCH)
    best=float("inf"); best_state=None; bad=0
    for ep in range(EPOCHS):
        model.train()
        for xb,yb in tr:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            l=loss_multi(model(xb),yb)
            l.backward(); opt.step()
        model.eval(); total=0; n=0
        with torch.no_grad():
            for xb,yb in va:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE)
                l=loss_multi(model(xb),yb)
                total+=l.item()*len(xb); n+=len(xb)
        val=total/n
        if val<best-1e-5:
            best=val; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

@torch.no_grad()
def evaluate_probe(model,X,y):
    dl=DataLoader(TensorDataset(X.float(),y),batch_size=PROBE_BATCH)
    pos_corr=None; exact=0; n=0
    for xb,yb in dl:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        pred=model(xb).argmax(-1)
        pc=(pred==yb).sum(0).cpu()
        pos_corr=pc if pos_corr is None else pos_corr+pc
        exact+=(pred==yb).all(1).sum().item()
        n+=len(xb)
    pos=(pos_corr.float()/n).numpy()
    return {
        "mean_pos_acc":float(pos.mean()),
        "exact_acc":float(exact/n),
        "first_pos_acc":float(pos[0]),
        "last_pos_acc":float(pos[-1]),
    }


In [ ]:

rows=[]

for model_key,model_id in MODELS.items():
    print("Loading",model_id)
    model=AutoModelForCausalLM.from_pretrained(
        model_id,cache_dir=str(CACHE_DIR),torch_dtype=DTYPE,low_cpu_mem_usage=True
    ).to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)

    for k in KS:
        for L in [max(1,k-1),k,k+1]:
            for seed in SEEDS:
                run_name=f"{model_key}_k{k}_L{L}_seed{seed}"
                done=CONTROL_DIR/f"{run_name}.json"
                if done.exists():
                    print("SKIP",run_name)
                    rows.append(json.load(open(done)))
                    continue

                set_seed(seed)
                P=generate_kernel(V,k,seed)
                tr_state,tr_obs,_=generate_dataset(P,V,k,L,N_TRAIN,seed+101)
                va_state,va_obs,_=generate_dataset(P,V,k,L,N_VAL,seed+202)
                te_state,te_obs,_=generate_dataset(P,V,k,L,N_TEST,seed+303)

                mapping=choose_token_ids(model,V,seed+int(hashlib.sha256(model_key.encode()).hexdigest()[:6],16))
                tr_ids=map_symbols(tr_obs,mapping)
                va_ids=map_symbols(va_obs,mapping)
                te_ids=map_symbols(te_obs,mapping)

                Xtr=extract_final(model,tr_ids,BATCH_EXTRACT[model_key])
                Xva=extract_final(model,va_ids,BATCH_EXTRACT[model_key])
                Xte=extract_final(model,te_ids,BATCH_EXTRACT[model_key])

                # 1. Visible-context reconstruction
                p_vis=train_probe(Xtr,tr_obs,Xva,va_obs,V)
                m_vis=evaluate_probe(p_vis,Xte,te_obs)

                # 2. Full-state reconstruction
                p_state=train_probe(Xtr,tr_state,Xva,va_state,V)
                m_state=evaluate_probe(p_state,Xte,te_state)

                # 3. Shuffled-label negative control
                rng=torch.Generator().manual_seed(seed+9999)
                perm=torch.randperm(len(tr_state),generator=rng)
                perm_val=torch.randperm(len(va_state),generator=rng)
                p_shuffle=train_probe(Xtr,tr_state[perm],Xva,va_state[perm_val],V)
                m_shuffle=evaluate_probe(p_shuffle,Xte,te_state)

                result={
                    "model_key":model_key,"model_id":model_id,"k":k,"L":L,"seed":seed,
                    "visible_exact_acc":m_vis["exact_acc"],
                    "visible_mean_pos_acc":m_vis["mean_pos_acc"],
                    "state_exact_acc":m_state["exact_acc"],
                    "state_mean_pos_acc":m_state["mean_pos_acc"],
                    "state_oldest_acc":m_state["first_pos_acc"],
                    "shuffle_state_exact_acc":m_shuffle["exact_acc"],
                    "shuffle_state_mean_pos_acc":m_shuffle["mean_pos_acc"],
                }

                json.dump(result,open(done,"w"),indent=2)
                rows.append(result)
                print(result)

                del P,Xtr,Xva,Xte,p_vis,p_state,p_shuffle
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

df=pd.DataFrame(rows)
df.to_csv(CSV_DIR/"part2_input_vs_state_control_raw.csv",index=False)
print("Completed rows:",len(df))


In [ ]:

summary=(
    df.groupby(["model_key","k","L"],as_index=False)
      .agg(
          visible_exact_mean=("visible_exact_acc","mean"),
          visible_exact_std=("visible_exact_acc","std"),
          state_exact_mean=("state_exact_acc","mean"),
          state_exact_std=("state_exact_acc","std"),
          oldest_acc_mean=("state_oldest_acc","mean"),
          shuffle_exact_mean=("shuffle_state_exact_acc","mean"),
          shuffle_pos_mean=("shuffle_state_mean_pos_acc","mean"),
          n=("seed","count"),
      )
)
summary["relative_context"]=summary["L"]-summary["k"]
summary.to_csv(CSV_DIR/"part2_input_vs_state_control_summary.csv",index=False)
display(summary)


In [ ]:
import matplotlib.pyplot as plt

In [ ]:

plt.figure(figsize=(8,5.5))
for mk in sorted(summary.model_key.unique()):
    d=summary[summary.model_key==mk].groupby("relative_context",as_index=False).agg(
        visible=("visible_exact_mean","mean"),
        state=("state_exact_mean","mean")
    )
    plt.plot(d.relative_context,d.visible,marker="o",label=f"{mk} visible-context")
    plt.plot(d.relative_context,d.state,marker="x",linestyle="--",label=f"{mk} full-state")

plt.axvline(0,linestyle="--",linewidth=1)
plt.xlabel(r"Relative context $L-k$")
plt.ylabel("Exact reconstruction accuracy")
plt.title("Visible input reconstruction vs full sufficient-state recovery")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR/"part2_control_visible_vs_state.png",dpi=300)
plt.savefig(FIG_DIR/"part2_control_visible_vs_state.pdf")
plt.show()


In [ ]:

# Integrity checks
expected=len(MODELS)*len(KS)*3*len(SEEDS)
print("Expected:",expected,"Rows:",len(df))
assert len(df)==expected, "Missing control runs"
assert df.isna().sum().sum()==0, "NaNs detected"

# Shuffled control should be near chance on average; exact state chance is V^-k.
print("Mean shuffled position accuracy:",df.shuffle_state_mean_pos_acc.mean())
print("Token-level chance:",1/V)

output=Path.cwd()/"Part2_InputVsState_Control_RESULTS.zip"
allowed={".csv",".json",".png",".pdf",".txt"}

with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED) as z:
    for f in CONTROL_DIR.rglob("*"):
        if f.is_file() and f.suffix.lower() in allowed:
            z.write(f,f.relative_to(PROJECT_ROOT))
    for f in [CSV_DIR/"part2_input_vs_state_control_raw.csv",
              CSV_DIR/"part2_input_vs_state_control_summary.csv",
              FIG_DIR/"part2_control_visible_vs_state.png",
              FIG_DIR/"part2_control_visible_vs_state.pdf"]:
        if f.exists():
            z.write(f,f.relative_to(PROJECT_ROOT))

print("ZIP:",output)
print("Size MB:",round(output.stat().st_size/1024**2,2))
